# Functions and Files

## Building reusable tools

Writing the same telemetry processing code over and over is tedious and error-prone. If you fix a bug in one place, you have to remember to fix it everywhere else. If you add a new check, you need to add it in every script that processes sol data.

This is the problem that **functions** solve. A function is a named, reusable block of code. Write it once, call it anywhere. And once we can write functions, we are ready to tackle a real task: reading data from files.

In this notebook, we will read actual CSV data from Curiosity's telemetry records.

## Defining functions

A function has three parts: a name, parameters (inputs), and a return value (output).

In [ ]:
def celsius_to_fahrenheit(celsius):
    """Convert a temperature from Celsius to Fahrenheit."""
    return (celsius * 9/5) + 32

# Call the function
print(celsius_to_fahrenheit(-63))   # -81.4
print(celsius_to_fahrenheit(0))     # 32.0
print(celsius_to_fahrenheit(-40))   # -40.0 (the one temperature where C == F)

Let's break that down:

- `def` tells Python we are defining a function.
- `celsius_to_fahrenheit` is the name. Like variables, use `snake_case`.
- `celsius` is the **parameter** -- a placeholder for the value we will pass in.
- The triple-quoted string is a **docstring** -- it documents what the function does. Get in the habit of writing these.
- `return` sends a value back to the caller.

In [ ]:
# Functions can have multiple parameters
def temperature_range(min_c, max_c):
    """Calculate the temperature range in Celsius."""
    return max_c - min_c

print(f"Range: {temperature_range(-63, -8)} degrees")

In [ ]:
# Default parameters provide sensible fallback values
def classify_wind(speed_ms, threshold=20):
    """Classify wind speed as safe or dangerous."""
    if speed_ms > threshold:
        return "dangerous"
    return "safe"

print(classify_wind(14.7))        # Uses default threshold of 20
print(classify_wind(14.7, 10))    # Override: threshold of 10
print(classify_wind(27.4))        # Clearly dangerous

In [ ]:
# Functions can return multiple values using tuples
def summarise_temps(temps):
    """Return the min, max, and average of a list of temperatures."""
    return min(temps), max(temps), sum(temps) / len(temps)

min_t, max_t, avg_t = summarise_temps([-63, -71, -58, -67, -74, -61, -69])
print(f"Min: {min_t}C, Max: {max_t}C, Avg: {avg_t:.1f}C")

## Pure functions vs side effects

A **pure function** takes inputs and returns outputs without changing anything else in the world. Given the same inputs, it always produces the same output. No surprises.

Data engineering loves pure functions, because they are predictable, testable, and composable. When a pipeline breaks at 3am, the last thing you want is functions that behave differently depending on some hidden state.

In [ ]:
# Pure function: depends only on its inputs, changes nothing
def is_anomalous(min_temp, wind_speed, status):
    """Check whether a sol's readings are anomalous."""
    return min_temp < -75 or wind_speed > 25 or status != "nominal"

# Always gives the same answer for the same inputs
print(is_anomalous(-63, 14.7, "nominal"))   # False
print(is_anomalous(-78, 14.7, "nominal"))   # True (cold)
print(is_anomalous(-63, 14.7, "warning"))   # True (status)

In [ ]:
# Side-effect function: changes something outside itself (prints, modifies a global)
anomaly_log = []

def log_anomaly(sol, reason):
    """Append an anomaly to the global log. (Side effect!)"""
    anomaly_log.append({"sol": sol, "reason": reason})
    print(f"Logged: Sol {sol} - {reason}")

log_anomaly(143, "Extreme cold")
log_anomaly(147, "High wind")
print(f"Log now has {len(anomaly_log)} entries")

The second function is not pure -- it modifies `anomaly_log` and prints to the screen. Sometimes side effects are necessary (you have to save data somewhere eventually), but keep them at the edges of your code. The core logic should be pure functions wherever possible.

## A more complete processing function

Let's write a function that processes a sol's data and returns a cleaned, typed dict.

In [ ]:
def process_sol(raw_dict):
    """Clean and type-convert a raw sol data dictionary.
    
    Parameters:
        raw_dict: dict with string values from CSV parsing
    
    Returns:
        dict with properly typed values, or None if critical data is missing
    """
    # Check for required fields
    if not raw_dict.get("sol"):
        return None
    
    processed = {
        "sol": int(raw_dict["sol"]),
        "earth_date": raw_dict.get("earth_date", "").strip(),
        "instrument_status": raw_dict.get("instrument_status", "unknown").strip().lower(),
    }
    
    # Convert numeric fields, handling missing data
    for field in ["min_temp_c", "max_temp_c", "pressure_pa", "wind_speed_ms"]:
        value = raw_dict.get(field, "").strip()
        processed[field] = float(value) if value else None
    
    uv = raw_dict.get("uv_index", "").strip()
    processed["uv_index"] = int(uv) if uv else None
    
    return processed

# Test it
raw = {"sol": "142", "earth_date": "2024-06-15", "min_temp_c": "-63",
       "max_temp_c": "-8", "pressure_pa": "735.4", "wind_speed_ms": "14.7",
       "uv_index": "7", "instrument_status": "nominal"}

result = process_sol(raw)
for key, value in result.items():
    print(f"  {key}: {value} ({type(value).__name__})")

## Reading files: the context manager

Now that we can process data, we need to *get* data. In practice, telemetry arrives in files -- CSVs, JSON, log files. Let's learn to read them.

But first, a cautionary tale.

In [ ]:
# The naive way to read a file (DON'T DO THIS)
# f = open("../data/mars_sol_sample.csv")
# content = f.read()
# print(content[:200])
# f.close()  # Easy to forget this line!

# What's wrong? If an error occurs between open() and close(),
# the file never gets closed. The operating system keeps a 
# "file descriptor" open, consuming resources. Do this enough
# times and your program leaks resources until it crashes.

The solution is the `with` statement, also called a **context manager**. It guarantees the file gets closed, even if an error occurs.

In [ ]:
# The correct way: use 'with'
with open("../data/mars_sol_sample.csv") as f:
    content = f.read()

# File is automatically closed when we leave the 'with' block
# Even if an error occurred inside, the file would still be closed

# Let's peek at the first few lines
lines = content.split("\n")
for line in lines[:5]:
    print(line)

The `with` statement is not optional politeness. It is how professionals handle files. If you see code that uses `open()` without `with`, treat it as a code smell. In data engineering, where pipelines process thousands of files, a forgotten `close()` is a ticking bomb.

## Reading CSV with the csv module

We could parse CSV by splitting on commas, but CSV has edge cases (quoted fields, embedded commas, newlines inside fields). Python's built-in `csv` module handles all of these correctly.

In [ ]:
import csv

# Read the CSV into a list of dicts
sol_data = []

with open("../data/mars_sol_sample.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        sol_data.append(row)

print(f"Loaded {len(sol_data)} rows")
print(f"Columns: {list(sol_data[0].keys())}")

In [ ]:
# Each row is a dict with string values
print("First row:")
for key, value in sol_data[0].items():
    print(f"  {key}: '{value}' ({type(value).__name__})")

Notice that every value is a string, even the numbers. This is exactly the problem we solved in our first notebook -- and exactly why we wrote that `process_sol` function.

In [ ]:
# Process all rows using our function
processed_data = []
skipped = 0

for raw_row in sol_data:
    result = process_sol(raw_row)
    if result is not None:
        processed_data.append(result)
    else:
        skipped += 1

print(f"Processed: {len(processed_data)} rows")
print(f"Skipped: {skipped} rows")
print(f"\nFirst processed row:")
for key, value in processed_data[0].items():
    display_type = type(value).__name__ if value is not None else "NoneType"
    print(f"  {key}: {value} ({display_type})")

In [ ]:
# Now we can do proper analysis
# Filter out None values for numeric calculations
valid_min_temps = [d["min_temp_c"] for d in processed_data if d["min_temp_c"] is not None]
valid_wind = [d["wind_speed_ms"] for d in processed_data if d["wind_speed_ms"] is not None]

print(f"Temperature stats ({len(valid_min_temps)} valid readings):")
print(f"  Coldest: {min(valid_min_temps):.1f}C")
print(f"  Warmest min: {max(valid_min_temps):.1f}C")
print(f"  Average: {sum(valid_min_temps) / len(valid_min_temps):.1f}C")

print(f"\nWind stats ({len(valid_wind)} valid readings):")
print(f"  Max: {max(valid_wind):.1f} m/s")
print(f"  Average: {sum(valid_wind) / len(valid_wind):.1f} m/s")

## Writing CSV files

Processing data is only half the story. We often need to write results back to files.

In [ ]:
# Find anomalous readings
anomalies = [
    d for d in processed_data
    if (d["min_temp_c"] is not None and d["min_temp_c"] < -75)
    or (d["wind_speed_ms"] is not None and d["wind_speed_ms"] > 25)
    or d["instrument_status"] not in ("nominal",)
]

print(f"Found {len(anomalies)} anomalous readings")
for a in anomalies[:5]:
    print(f"  Sol {a['sol']}: temp={a['min_temp_c']}, wind={a['wind_speed_ms']}, status={a['instrument_status']}")

In [ ]:
# Write anomalies to a new CSV
output_path = "../data/anomalies_sample.csv"
fieldnames = ["sol", "earth_date", "min_temp_c", "wind_speed_ms", "instrument_status"]

with open(output_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(anomalies)

print(f"Wrote {len(anomalies)} anomalies to {output_path}")

## Working with JSON

CSV is great for tabular data, but sometimes we need a more flexible format. **JSON** (JavaScript Object Notation) can represent nested structures, and it maps directly to Python dicts and lists.

JSON is everywhere in data engineering: API responses, configuration files, log entries, message queues.

In [ ]:
import json

# Convert Python data to a JSON string
sol_summary = {
    "report_type": "weekly_summary",
    "sol_range": [142, 148],
    "stats": {
        "avg_min_temp": -66.1,
        "avg_wind_speed": 16.1,
        "anomaly_count": 3
    },
    "anomalous_sols": [143, 144, 147]
}

json_string = json.dumps(sol_summary, indent=2)
print(json_string)

In [ ]:
# Parse a JSON string back into Python objects
parsed = json.loads(json_string)
print(f"Type: {type(parsed)}")
print(f"Anomaly count: {parsed['stats']['anomaly_count']}")
print(f"Anomalous sols: {parsed['anomalous_sols']}")

In [ ]:
# Write JSON to a file
output_path = "../data/sol_summary.json"

with open(output_path, "w") as f:
    json.dump(sol_summary, f, indent=2)

print(f"Saved summary to {output_path}")

In [ ]:
# Read JSON from a file
with open(output_path) as f:
    loaded = json.load(f)

print(f"Loaded: {loaded['report_type']}")
print(f"Sol range: {loaded['sol_range']}")

A quick reference for the json module:

| Function | Direction | Input | Output |
|----------|-----------|-------|--------|
| `json.dumps()` | Python -> string | dict/list | JSON string |
| `json.loads()` | string -> Python | JSON string | dict/list |
| `json.dump()` | Python -> file | dict/list + file | writes to file |
| `json.load()` | file -> Python | file | dict/list |

The `s` in `dumps`/`loads` stands for "string". With the `s`, you work with strings. Without the `s`, you work with files.

## Error handling with try/except

Real data is messy. Files go missing. Values are corrupted. Network connections drop. Robust code anticipates failures and handles them gracefully, rather than crashing with an inscrutable traceback.

In [ ]:
# What happens when a file doesn't exist?
try:
    with open("../data/nonexistent_file.csv") as f:
        data = f.read()
except FileNotFoundError:
    print("File not found! Using default data instead.")
    data = ""

In [ ]:
# Handling bad data during conversion
raw_values = ["-63", "N/A", "-58", "", "-74", "error", "-69"]

clean_values = []
for val in raw_values:
    try:
        clean_values.append(float(val))
    except ValueError:
        print(f"  Skipping invalid value: '{val}'")

print(f"\nClean values: {clean_values}")
print(f"Kept {len(clean_values)} of {len(raw_values)} values")

In [ ]:
# A robust file-reading function
def load_sol_data(filepath):
    """Load and process sol data from a CSV file.
    
    Returns a list of processed dicts, or an empty list if the file
    cannot be read.
    """
    try:
        with open(filepath) as f:
            reader = csv.DictReader(f)
            raw_rows = list(reader)
    except FileNotFoundError:
        print(f"Error: File not found: {filepath}")
        return []
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return []
    
    processed = []
    for row in raw_rows:
        result = process_sol(row)
        if result is not None:
            processed.append(result)
    
    print(f"Loaded {len(processed)} rows from {filepath}")
    return processed

# Test with a real file
data = load_sol_data("../data/mars_sol_sample.csv")

# Test with a missing file
empty = load_sol_data("../data/does_not_exist.csv")

## Bringing it all together

Let's build a small but complete data processing pipeline: read a CSV, process the data, generate a summary, and save it as JSON.

In [ ]:
def safe_mean(values):
    """Calculate the mean of a list, ignoring None values."""
    valid = [v for v in values if v is not None]
    if not valid:
        return None
    return sum(valid) / len(valid)


def generate_summary(sol_records):
    """Generate a summary report from processed sol data."""
    if not sol_records:
        return {"error": "No data to summarise"}
    
    min_temps = [r["min_temp_c"] for r in sol_records]
    max_temps = [r["max_temp_c"] for r in sol_records]
    winds = [r["wind_speed_ms"] for r in sol_records]
    statuses = [r["instrument_status"] for r in sol_records]
    
    # Count statuses
    status_counts = {}
    for s in statuses:
        status_counts[s] = status_counts.get(s, 0) + 1
    
    return {
        "total_sols": len(sol_records),
        "sol_range": [sol_records[0]["sol"], sol_records[-1]["sol"]],
        "avg_min_temp_c": round(safe_mean(min_temps), 1) if safe_mean(min_temps) else None,
        "avg_max_temp_c": round(safe_mean(max_temps), 1) if safe_mean(max_temps) else None,
        "avg_wind_speed_ms": round(safe_mean(winds), 1) if safe_mean(winds) else None,
        "instrument_status_counts": status_counts,
        "missing_data_count": sum(1 for r in sol_records 
                                   if any(r[k] is None for k in ["min_temp_c", "max_temp_c", "wind_speed_ms"])),
    }


# Run the pipeline
data = load_sol_data("../data/mars_sol_sample.csv")
summary = generate_summary(data)

print(json.dumps(summary, indent=2))

In [ ]:
# Save the summary
with open("../data/pipeline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Pipeline complete. Summary saved.")

---

## Exercises

### Exercise 1: Write a conversion function

Write a function called `pascals_to_millibars` that converts atmospheric pressure from Pascals to millibars. The conversion is: `1 millibar = 100 Pascals`.

Test it with the value 735.4 Pa (should return approximately 7.354 mbar).

In [ ]:
# Your code here


### Exercise 2: Anomaly classifier

Write a function called `classify_sol` that takes a processed sol dict and returns a string classification:
- `"critical"` if the instrument is offline OR wind exceeds 30 m/s
- `"warning"` if temp is below -75C OR wind exceeds 20 m/s OR status is "warning"
- `"nominal"` otherwise

Handle `None` values gracefully (treat missing readings as nominal for that field).

Test it with at least three different inputs.

In [ ]:
# Your code here


### Exercise 3: Read and process

1. Read `../data/mars_sol_sample.csv` using `csv.DictReader`
2. Process each row to convert numeric fields (handle missing values!)
3. Find the 5 sols with the highest wind speeds
4. Print them in a formatted table

In [ ]:
# Your code here


### Exercise 4: JSON report generator

Write a function called `generate_json_report` that:
1. Takes a list of processed sol dicts
2. Calculates: total sols, number of anomalous sols, average temperature, date range
3. Returns the report as a JSON-serialisable dict
4. Saves it to `../data/my_report.json`

Use the data loaded from `mars_sol_sample.csv`.

In [ ]:
# Your code here


### Exercise 5: Robust file processor

Write a function called `process_telemetry_file` that:
1. Takes a file path as an argument
2. Handles `FileNotFoundError` gracefully
3. Reads and processes the CSV data
4. Counts how many rows had missing data in any numeric field
5. Returns a tuple of `(processed_rows, error_count)`

Test it with both `../data/mars_sol_sample.csv` (exists) and `../data/fake_file.csv` (does not exist).

In [ ]:
# Your code here


---

## Summary

This notebook covered the tools that turn one-off scripts into proper data processing code:

- **Functions** (`def`, `return`, parameters, defaults) make code reusable and testable
- **Pure functions** are predictable and composable -- prefer them over functions with side effects
- **The `with` statement** guarantees files are closed properly. Always use it.
- **`csv.DictReader`** reads CSV files into a list of dicts with proper handling of edge cases
- **`json.dumps`/`json.loads`** convert between Python objects and JSON strings
- **`try`/`except`** lets code fail gracefully instead of crashing

We have now built a small but genuine data pipeline: read a file, clean the data, compute statistics, save results. But we are doing a lot of work that has already been done for us by a library called pandas. Next up, we will see how pandas can replace dozens of lines of our hand-rolled code with a single, expressive line.